In [ ]:
# @title
import os,subprocess,sys,time,requests,json,re,socket,urllib.request
import threading
from IPython.display import clear_output, display, HTML, Image

check_value = {
    'User_folder' : '/content/drive/MyDrive/SD-Data',
    'SDVNFolder': '/content/SDVN-WebUI',
    'CommandLine' : '',
    'Sever_Pinggy': '',
}
for key, value in check_value.items():
  if key not in globals():
    globals()[key] = value
  elif type(globals()[key]) == str:
    globals()[key] = globals()[key].split(' ')[-1]

root_folder = '/content'
Version_folder = f"{root_folder}/ComfyUI"

def c_folder(path):
  path = f'{User_folder}/{path}'
  os.makedirs(path, exist_ok=True)

def link_folder(source,target):
  !rm -rf {target}
  !ln -s {source} {target}

# --- Tunneling Functions ---
def check_cloudflare_setting():
    domain_setting = f"{User_folder}/Setting/Domain_sever.txt"
    if not os.path.exists(domain_setting): return [None, None]
    with open(domain_setting, "r", encoding="utf-8") as file:
        lines = file.readlines()
    output = [None, None]
    for line in lines:
        if "cloudflare" in line and "#" not in line:
            flare_token = line.split("-")[-1].strip()
            flare_domain = line.split("-")[1].strip()
            output = [flare_token, flare_domain]
    return output
            
def cloudflare_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      if sock.connect_ex(('127.0.0.1', port)) == 0: break
      sock.close()
  flare_token, flare_domain = check_cloudflare_setting()
  if flare_token != None:
    subprocess.Popen(["cloudflared", "tunnel", "run", "--token", flare_token])
    print(f"\033[92m{'🔗 Link online để sử dụng (custom flare):'}\033[0m", f"https://{flare_domain}")
  p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print(f"\033[92m{'🔗 Link online để sử dụng:'}\033[0m", l[l.find("http"):], end='')

def pinggy_thread(port,pinggy):
  server = {"Auto": "", "USA": "us.", "Europe": "eu.", "Asia": "ap.", "South America": "br.", "Australia": "au."}
  sv = server.get(Sever_Pinggy, "")
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      if sock.connect_ex(('127.0.0.1', port)) == 0: break
      sock.close()
  try:
    if pinggy != None:
      if ":" in pinggy:
          pinggy, ac, ps = pinggy.split(":")
          cmd = ["ssh", "-p", "443", f"-R0:localhost:{port}", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=30", f"{pinggy}@{sv}pro.pinggy.io", f'\"b:{ac}:{ps}\"']
      else:
          cmd = ["ssh", "-p", "443", f"-R0:localhost:{port}", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=30", f"{pinggy}@{sv}pro.pinggy.io"]
    else:
      cmd = ["ssh", "-p", "443", "-L4300:localhost:4300", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=30", f"-R0:localhost:{port}", "free.pinggy.io"]
    process = subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True)
    for line in iter(process.stdout.readline, ''):
        match = re.search(r'(https?://[^\s]+)', line)
        if match:
            url = match.group(1)
            if "dashboard.pinggy.io" in url: continue
            print(f"\033[92m🔗 Link online để sử dụng:\033[0m {url}")
            if pinggy == None:
              display(HTML("<div><code style='color:yellow'>Link pinggy free hoạt động trong 60phút</code></div>"))
            break
  except Exception as e: print(f"❌ Lỗi: {e}")

def sever_flare(port, api = None, pinggy = None):
  threading.Thread(target=pinggy_thread, daemon=True, args=(port,pinggy,)).start()
  threading.Thread(target=cloudflare_thread, daemon=True, args=(port,)).start()

# --- Setup Folders ---
folder_list = ['Model', 'Lora', 'Embeddings', 'ControlnetModel', 'ComfyUIinput', 'Setting']
for i in folder_list: c_folder(i)

html="<h1 style='color:lightgreen; font-size:3em' ><code>BẮT ĐẦU CÀI ĐẶT COMFYUI CƠ BẢN - VUI LÒNG ĐỢI ...</code></h1>"
display(HTML(html))

# --- Install ComfyUI ---
%cd {root_folder}
if not os.path.exists(Version_folder):
    !git clone https://github.com/Comfy-Org/ComfyUI
!pip install -q -r {Version_folder}/requirements.txt

# --- Install ComfyUI Manager ---
%cd {Version_folder}/custom_nodes
if not os.path.exists("ComfyUI-Manager"):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager
    !pip install -q -r ComfyUI-Manager/requirements.txt

# --- Link Folders ---
list_link_folder = {
    'Model':'models/checkpoints/Model',
    'Lora':'models/loras/Lora',
    'ControlnetModel':'models/controlnet',
    'Setting/Comfy_Setting':'user'
}
for key, value in list_link_folder.items():
  link_folder(f'{User_folder}/{key}',f'{Version_folder}/{value}')

!cp -f {SDVNFolder}/ComfySetting/extra_model_paths.yaml {Version_folder}/extra_model_paths.yaml 2>/dev/null || true


# --- Cài đặt Custom Nodes ---
%cd /content/ComfyUI/custom_nodes

import os

# Danh sách các node
nodes = [
    "https://github.com/rgthree/rgthree-comfy",
    "https://github.com/kijai/ComfyUI-KJNodes",
    "https://github.com/yolain/ComfyUI-Easy-Use",
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite",
    "https://github.com/cubiq/ComfyUI_essentials",
    "https://github.com/christian-byrne/audio-separation-nodes-comfyui",
    "https://github.com/akatz-ai/ComfyUI-Basic-Math",
    "https://github.com/Derfuu/Derfuu_ComfyUI_ModdedNodes",
    "https://github.com/kijai/ComfyUI-MelBandRoFormer",
    "https://github.com/M1kep/ComfyLiterals",
    "https://github.com/evanspearman/ComfyMath",
    "https://github.com/crystian/ComfyUI-Crystools",
    "https://github.com/kijai/ComfyUI-PromptRelay",
    "https://github.com/kijai/ComfyUI-WanAnimatePreprocess",
    "https://github.com/kustomzone/ComfyUI_SoundFlow"
]

for node in nodes:
    folder_name = node.split('/')[-1]
    if not os.path.exists(folder_name):
        !git clone --depth 1 {node}

# Cài đặt tất cả dependencies trong 1 lần cho nhanh
!pip install sageattention triton
!pip install -r /content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite/requirements.txt
!pip install -r /content/ComfyUI/custom_nodes/ComfyUI-MelBandRoFormer/requirements.txt
!pip install -r /content/ComfyUI/custom_nodes/ComfyMath/requirements.txt
!pip install -r /content/ComfyUI/custom_nodes/ComfyUI-Crystools/requirements.txt
!pip install -r /content/ComfyUI/custom_nodes/ComfyUI-PromptRelay/requirements.txt
!pip install -r /content/ComfyUI/custom_nodes/ComfyUI-WanAnimatePreprocess/requirements.txt

# FIX LỖI onnxruntime-gpu libcudnn.so.9 TRÊN COLAB
!pip install onnxruntime-gpu==1.18.1

# --- BƯỚC 3: CẤU HÌNH ĐƯỜNG DẪN MODEL TỪ DRIVE ---
import yaml

base_drive_path = "/content/drive/MyDrive/SD-Data-Share/ComfyModel"

config = {
    "comfyui": {
        "base_path": base_drive_path,
        "checkpoints": "checkpoints",
        "clip": "clip",
        "clip_vision": "clip_vision",
        "diffusers": "diffusers",
        "diffusion_models": "diffusion_models",
        "vae": "vae",
        "loras": "loras",
        "controlnet": "controlnet",
        "upscale_models": "upscale_models",
        "latent_upscale_models": "latent_upscale_models",
        "animatediff_models": "animatediff_models",
        "embeddings": "embeddings",
        "ipadapter": "ipadapter",
        "instantid": "instantid",
        "audio_encoders": "audio_encoders",
        "style_models": "style_models",
        "detection": "detection",
        "pulid": "pulid",
        "facerestore_models": "facerestore_models"
    }
}

with open('/content/ComfyUI/extra_model_paths.yaml', 'w') as f:
    yaml.dump(config, f)


clear_output()
display(HTML("<h1 style='color:lightgreen'><code>CÀI ĐẶT HOÀN TẤT! </code></h1>"))


# --- Run ComfyUI ---
%cd {Version_folder}
pinggy = None
domain_setting = f"{User_folder}/Setting/Domain_sever.txt"
if os.path.exists(domain_setting):
    with open(domain_setting, "r", encoding="utf-8") as file:
        for line in file.readlines():
            if "pinggy" in line and "#" not in line:
                pinggy = line.split("-")[1].strip()

sever_flare(8888, None, pinggy)
final_arg = f"--preview-method auto --port 8888 --input-directory {User_folder}/ComfyUIinput {CommandLine}"
!python main.py {final_arg}
